# Step 2: Clean, split and build the YOLO dataset
Produces `data/yolo/{train,val,test}/{images,labels}` and `data/yolo/data.yaml`, plus two CSVs in `data/` that record exactly what was excluded and which split each image went to.

Rules: images flagged in step 1 are excluded; the split is **stratified by class** (70/20/10, seed 0); byte-identical duplicates stay together.

In [ ]:
# Set True for a fast CPU sanity run (3 epochs, 320 px, 25% of train). False = full experiment.
QUICK_TEST = False
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # works from VS Code either way
sys.path.insert(0, str(ROOT / "src"))
import mri_detection as md
from mri_detection import data, viz, models, train, evaluate
cfg = md.load_config(quick=QUICK_TEST)
RUNS = md.ROOT / "runs"
YOLO_DIR = md.ROOT / cfg["dataset"]["yolo_dir"]
DATA_YAML = YOLO_DIR / "data.yaml"

In [ ]:
root = data.extract_zip(md.ROOT / cfg["dataset"]["raw_zip"], md.ROOT / cfg["dataset"]["raw_dir"])
df, folder2id, names = data.clean_dataset(data.scan_dataset(root))
s = cfg["dataset"]["split"]
df = data.split_dataset(df, (s["train"], s["val"], s["test"]), seed=cfg["seed"])
print("Excluded:", df.issue.notna().sum(), "images ->", df.issue.value_counts().to_dict())

In [ ]:
data_dir = md.ROOT / "data"
rel = lambda p: str(Path(p).relative_to(root))
export = df.assign(image=df.image.map(rel))[["image", "folder", "n_boxes", "issue", "split"]]
export[export.issue.notna()].to_csv(data_dir / "excluded_images.csv", index=False)
export[export.split.notna()].to_csv(data_dir / "split_manifest.csv", index=False)
print("Saved data/excluded_images.csv and data/split_manifest.csv")

In [ ]:
data_yaml = data.build_yolo_dataset(df, names, YOLO_DIR)
print("Wrote", data_yaml)
summary = data.split_summary(df, names)
summary["total boxes"] = summary[names].sum(axis=1)
summary

## Check: the written files match the plan

In [ ]:
import yaml
d = yaml.safe_load(open(data_yaml))
print("names:", d["names"])
for sp in ["train", "val", "test"]:
    n_img = len(list((YOLO_DIR / sp / "images").iterdir())); n_lbl = len(list((YOLO_DIR / sp / "labels").glob("*.txt")))
    print(f"{sp:5s} images={n_img:5d} labels={n_lbl:5d}")
    assert n_img == n_lbl == (df.split == sp).sum()
tr = set(df[df.split == "train"].md5); others = set(df[df.split.isin(["val", "test"])].md5)
print("Images identical to a training image found in val/test:", len(tr & others))
assert not tr & others

In [ ]:
import matplotlib.pyplot as plt, random
random.seed(1)
files = random.sample(sorted((YOLO_DIR / "train" / "images").iterdir()), 8)
imgs = [viz.draw_boxes(f, viz.read_yolo_labels(YOLO_DIR / "train" / "labels" / (f.stem + ".txt")), names) for f in files]
viz.show_grid(imgs, ncols=4, figsize=(14, 3.6)); plt.show()

### Limitation to keep in mind
The split is per image. The dataset gives no patient ids, so slices from one patient can land in different splits and make scores look better than they would on new patients. Report this in the thesis/paper.

Next: `03_baseline_yolov8m.ipynb`.